# DSM - Example 4 (hour 1 sub-problem)

**Original AMPL author:** Xingpeng Li - Associate Professor, Dept. of Electrical and Computer Engineering, University of Houston (UH), Houston, TX, USA (Senior Member, IEEE). Email: xli83@central.uh.edu

**Converted by:** Haoxiang Wan - PhD student of Dr. Xingpeng Li (AMPL -> Pyomo + Gurobi)

Solves the first hour; the deferred amount feeds the hour-2 model.

In [1]:
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, Objective, Constraint, SolverFactory,
    Binary, NonNegativeReals, minimize, value
)

# ---- Data: loaded from external file 'DSM_IC_e4_hr1_data.txt' ----
import sys, pathlib
# Make the ampl_data parser importable (one level up from this notebook)
_pkg = pathlib.Path.cwd().parent
if str(_pkg) not in sys.path: sys.path.insert(0, str(_pkg))
from ampl_data import parse_ampl_data

_d = parse_ampl_data('DSM_IC_e4_hr1_data.txt')

GEN_data     = _d['GEN']
PERIOD_data  = _d['PERIOD']
gen_min      = _d['gen_min']
gen_max      = _d['gen_max']
gen_RRlimit  = _d['gen_RRlimit']
gen_OpCost   = _d['gen_OpCost']
gen_NlCost   = _d['gen_NlCost']
gen_SuCost   = _d['gen_SuCost']
Time_TotalPd = _d['Time_TotalPd']
DSM_d_Limit  = _d['DSM_d_Limit']

DSM_Cost = 25

m = ConcreteModel()
m.GEN    = Set(initialize=GEN_data, ordered=True)
m.PERIOD = Set(initialize=PERIOD_data, ordered=True)

m.gen_min     = Param(m.GEN, initialize=gen_min)
m.gen_max     = Param(m.GEN, initialize=gen_max)
m.gen_RRlimit = Param(m.GEN, initialize=gen_RRlimit)
m.gen_OpCost  = Param(m.GEN, initialize=gen_OpCost)
m.gen_NlCost  = Param(m.GEN, initialize=gen_NlCost)
m.gen_SuCost  = Param(m.GEN, initialize=gen_SuCost)

m.Time_TotalPd = Param(m.PERIOD, initialize=Time_TotalPd)
m.DSM_d_Limit  = Param(m.PERIOD, initialize=DSM_d_Limit)

m.u  = Var(m.GEN, m.PERIOD, domain=Binary)
m.v  = Var(m.GEN, m.PERIOD, domain=Binary)
m.Pg = Var(m.GEN, m.PERIOD)
m.DSM_d = Var(m.PERIOD, domain=NonNegativeReals)

m.obj = Objective(
    rule=lambda mm: sum(mm.gen_OpCost[g]*mm.Pg[g,t]
                       + mm.gen_NlCost[g]*mm.u[g,t]
                       + mm.gen_SuCost[g]*mm.v[g,t]
                       for g in mm.GEN for t in mm.PERIOD)
       + sum(mm.DSM_d[t]*DSM_Cost for t in mm.PERIOD),
    sense=minimize
)

m.PowerBalance = Constraint(expr=sum(m.Pg[g,1] for g in m.GEN) == m.Time_TotalPd[1] - m.DSM_d[1])
m.genLimit_Min = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.gen_min[g]*mm.u[g,t] <= mm.Pg[g,t])
m.genLimit_Max = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.Pg[g,t] <= mm.gen_max[g]*mm.u[g,t])

def vu_rule(mm,g,t):
    if t == mm.PERIOD.first():
        return mm.v[g,t] >= mm.u[g,t]
    return mm.v[g,t] >= mm.u[g,t] - mm.u[g, mm.PERIOD.prev(t)]
m.genVU = Constraint(m.GEN, m.PERIOD, rule=vu_rule)

m.DSM_limit  = Constraint(m.PERIOD, rule=lambda mm,t: mm.DSM_d[t] <= mm.DSM_d_Limit[t])
m.unit_limit = Constraint(m.GEN, m.PERIOD, rule=lambda mm,g,t: mm.u[g,t] == 1)

model = m

In [2]:
# ---- Solve with Gurobi ----
solver = SolverFactory('gurobi')
solver.options['MIPGap'] = 0.0
solver.options['TimeLimit'] = 90
results = solver.solve(model, tee=True)
print(results.solver.status, results.solver.termination_condition)
m = model
print("g  t   v       u    Pg")
for g in m.GEN:
    for t in m.PERIOD:
        print(f"{g}  {t}   {value(m.v[g,t]):.2f}   {int(round(value(m.u[g,t])))}   {value(m.Pg[g,t]):.3f}")
if hasattr(m, "DSM_d"):
    print("\nDSM deferred load:")
    for t in m.PERIOD:
        print(f"  t={t}  DSM_d = {value(m.DSM_d[t]):.3f}")

Read LP format model from file C:\Users\hwan6\AppData\Local\Temp\tmpobfz817s.pyomo.lp


Reading time = 0.00 seconds
x1: 10 rows, 7 columns, 18 nonzeros
Set parameter MIPGap to value 0
Set parameter TimeLimit to value 90
Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 12th Gen Intel(R) Core(TM) i7-12700, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 20 logical processors, using up to 20 threads

Non-default parameters:
TimeLimit  90
MIPGap  0

Optimize a model with 10 rows, 7 columns and 18 nonzeros
Model fingerprint: 0x30d59c19


Variable types: 3 continuous, 4 integer (4 binary)
Coefficient statistics:


  Matrix range     [1e+00, 9e+01]
  Objective range  [1e+01, 8e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+02]


Presolve removed 9 rows and 4 columns
Presolve time: 0.00s


...

Variable types: 0 continuous, 3 integer (0 binary)

Root relaxation: objective 2.700000e+03, 1 iterations, 0.00 seconds (0.00 work units)


    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    2700.0000000 2700.00000  0.00%     -    0s



Explored 1 nodes (1 simplex iterations) in 0.00 seconds (0.00 work units)


Thread count was 20 (of 20 available processors)

Solution count 1: 2700 

Optimal solution found (tolerance 0.00e+00)
Best objective 2.700000000000e+03, best bound 2.700000000000e+03, gap 0.0000%


ok optimal
g  t   v       u    Pg
1  1   1.00   1   80.000
2  1   1.00   1   20.000

DSM deferred load:
  t=1  DSM_d = 10.000
